In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

In [7]:
os.environ["GROQ_API_KEY"] = "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"


In [8]:
pdf_path = "Reis_Cleaning_Services_Guide.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"Loaded {len(documents)} pages")

Loaded 3 pages


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 6 chunks


In [10]:
# 3. Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'trust_remote_code': True}
)

# 4. Create FAISS vector store
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector store created")

Vector store created


In [11]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)


In [12]:
# ------------------------------
# Section 1: Define Prompt
# ------------------------------

from langchain_core.prompts import ChatPromptTemplate

cleaning_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a professional cleaning services assistant. 
Rules:
1. Answer the customer's question using ONLY the provided context (cleaning services documents).
2. Respond clearly and professionally with detailed explanations.
3. After answering, ask: "Would you like to place an order for our services?"
4. If customer says 'yes':
   - Immediately ask for their name and phone number.
   - When received, return ONLY valid JSON:
     {{"name": "", "phone_number": "", "service_requested": "", "notes": ""}}
   - Use the original question as "service_requested" and notes for any additional comments.
5. Do NOT include explanations when returning JSON.
"""),
    ("human", "Context:\n{context}\n\nCustomer Question:\n{input}")
])

In [13]:
# ------------------------------
# Section 2: Email & Ticket Functions
# ------------------------------

import os
import json
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- Email Configuration ---
SMTP_SENDER_EMAIL    = "zimorworld@gmail.com"
SMTP_SENDER_PASSWORD = "gasiitmnnkvjvtac"
SMTP_AGENT_EMAIL     = "pairuoyefe@gmail.com"
SMTP_HOST            = "smtp.gmail.com"
SMTP_PORT            = 587
TICKET_FILE          = "support_ticket.json"


def send_ticket_email(ticket_data):
    """Send ticket JSON as email to agent"""
    msg = MIMEMultipart()
    msg["From"] = SMTP_SENDER_EMAIL
    msg["To"] = SMTP_AGENT_EMAIL
    msg["Subject"] = f"New Cleaning Service Ticket: {ticket_data['service_requested']}"
    body = json.dumps(ticket_data, indent=4)
    msg.attach(MIMEText(body, "plain"))

    with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
        server.starttls()
        server.login(SMTP_SENDER_EMAIL, SMTP_SENDER_PASSWORD)
        server.sendmail(SMTP_SENDER_EMAIL, SMTP_AGENT_EMAIL, msg.as_string())
    print("Ticket sent via email.")

In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def handle_customer_query(query, vectorstore, llm):
    """Retrieve context from vectorstore, invoke LLM, and handle order ticket"""
    # 1️⃣ Retrieve context from vectorstore
    docs_with_scores = vectorstore.similarity_search_with_score(query, k=3)
    context = "\n\n".join([doc.page_content for doc, _ in docs_with_scores]) if docs_with_scores else ""

    # 2️⃣ Invoke LLM with the prompt
    chain = cleaning_prompt | llm
    response = chain.invoke({"context": context, "input": query})
    answer = response.content
    print("LLM Response:\n", answer)

    # 3️⃣ Ask customer about placing an order
    if "would you like to place an order" in answer.lower():
        # --- Use widgets instead of input() ---
        name_field = widgets.Text(placeholder="Enter your name", description="Name:")
        phone_field = widgets.Text(placeholder="Enter phone number", description="Phone:")
        submit_btn = widgets.Button(description="Submit Order", button_style="success")
        cancel_btn = widgets.Button(description="No Thanks", button_style="danger")
        out = widgets.Output()

        display(widgets.VBox([name_field, phone_field, submit_btn, cancel_btn, out]))

        def on_submit(b):
            with out:
                clear_output()
                name = name_field.value.strip()
                phone = phone_field.value.strip()
                if not name or not phone:
                    print("⚠️ Please fill in both fields.")
                    return
                ticket_data = {
                    "name": name,
                    "phone_number": phone,
                    "service_requested": query,
                    "notes": ""
                }
                with open(TICKET_FILE, "w", encoding="utf-8") as f:
                    json.dump(ticket_data, f, ensure_ascii=False, indent=4)
                send_ticket_email(ticket_data)
                print(f"✅ Order placed for {name}! Ticket saved and email sent.")

        def on_cancel(b):
            with out:
                clear_output()
                print("❌ Order cancelled.")

        submit_btn.on_click(on_submit)
        cancel_btn.on_click(on_cancel)

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def handle_customer_query(query, vectorstore, llm):
    """Chat-style interactive order flow"""
    
    # 1️⃣ Retrieve context
    docs_with_scores = vectorstore.similarity_search_with_score(query, k=3)
    context = "\n\n".join([doc.page_content for doc, _ in docs_with_scores]) if docs_with_scores else ""

    # 2️⃣ Invoke LLM
    chain = cleaning_prompt | llm
    response = chain.invoke({"context": context, "input": query})
    answer = response.content
    print("🤖 Reis Cleaning:\n", answer)

    # 3️⃣ Only trigger order flow if LLM asks
    if "would you like to place an order" in answer.lower():
        
        out = widgets.Output()
        display(out)
        
        # --- State tracker ---
        state = {"step": "confirm", "name": "", "phone": ""}
        
        chat_input = widgets.Text(
            placeholder="Type your reply here...",
            layout=widgets.Layout(width="400px")
        )
        send_btn = widgets.Button(description="Send ➤", button_style="primary")
        display(widgets.HBox([chat_input, send_btn]))

        def next_step(b):
            user_text = chat_input.value.strip()
            chat_input.value = ""

            with out:
                if state["step"] == "confirm":
                    if user_text.lower() in ["yes", "yeah", "sure", "yep", "ok", "okay"]:
                        print(f"👤 You: {user_text}")
                        print("🤖 Reis Cleaning: Great! What is your full name?")
                        state["step"] = "name"
                    else:
                        print(f"👤 You: {user_text}")
                        print("🤖 Reis Cleaning: No problem! Feel free to reach out anytime. 😊")
                        send_btn.disabled = True
                        chat_input.disabled = True

                elif state["step"] == "name":
                    state["name"] = user_text
                    print(f"👤 You: {user_text}")
                    print("🤖 Reis Cleaning: Got it! What's your phone number?")
                    state["step"] = "phone"

                elif state["step"] == "phone":
                    state["phone"] = user_text
                    print(f"👤 You: {user_text}")
                    print("🤖 Reis Cleaning: Perfect! What type of service do you need? (e.g. Deep Cleaning, Regular Cleaning, Move-in/out)")
                    state["step"] = "service"

                elif state["step"] == "service":
                    service = user_text
                    print(f"👤 You: {user_text}")

                    ticket_data = {
                        "name": state["name"],
                        "phone_number": state["phone"],
                        "service_requested": service,
                        "notes": ""
                    }

                    with open(TICKET_FILE, "w", encoding="utf-8") as f:
                        json.dump(ticket_data, f, ensure_ascii=False, indent=4)

                    send_ticket_email(ticket_data)

                    print(f"\n✅ Order confirmed for {state['name']}!")
                    print(f"   📞 Phone: {state['phone']}")
                    print(f"   🧹 Service: {service}")
                    print("🤖 Reis Cleaning: Thank you! We'll be in touch soon. 🙌")

                    send_btn.disabled = True
                    chat_input.disabled = True

        send_btn.on_click(next_step)

In [18]:
query = "what services do you offer"
handle_customer_query(query, vectorstore, llm)


🤖 Reis Cleaning:
 At Reis Cleaning Services, we offer a wide range of facility care and hygiene solutions, including:

1. Post-Construction & Renovation Cleanup: This service involves the complete removal of construction debris, dust, paint marks, and residues, with final floor prep, wall wipe-downs, window cleaning, and move-in readiness.
2. Environmental Health & Pest Control: We offer certified fumigation for homes and food environments, termite, rodent, bedbug, and general pest eradication, as well as hygienic disinfection for high-risk sites.
3. Consultancy & Training Services: Our services include recruitment, training, and quality control systems for cleaning staff, as well as customized facility management frameworks for estates and corporations.
4. Residential Services: We offer basic cleaning and deep cleaning services for various property types, including standard rates for rooms, flats, and duplexes.

Would you like to place an order for our services?


Output()